## Step 1 — Install Dependencies

In [ ]:
from google.colab import files
from google.colab import files
import os
import subprocess
from ultralytics import YOLO
import cv2
import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import base64
from IPython.display import HTML, display


In [ ]:
!pip install ultralytics supervision opencv-python-headless --quiet
print(' Dependencies installed')

 Dependencies installed


## Step 2 — Upload Your Two Videos

In [ ]:
print(' Upload your two MP4 videos (select both at once or one by one):')
uploaded = files.upload()

video_paths = []
for fname in uploaded:
    if fname.lower().endswith('.mp4'):
        video_paths.append(fname)
        print(f'   {fname}  ({os.path.getsize(fname)/1024:.1f} KB)')

if len(video_paths) < 2:
    print('  Please upload exactly 2 MP4 files.')
else:
    VIDEO_1 = video_paths[0]
    VIDEO_2 = video_paths[1]
    print(f'\nVideo 1: {VIDEO_1}')
    print(f'Video 2: {VIDEO_2}')

 Upload your two MP4 videos (select both at once or one by one):


Saving istockphoto-181843093-640_adpp_is.mp4 to istockphoto-181843093-640_adpp_is (1).mp4
Saving istockphoto-181843094-640_adpp_is (1).mp4 to istockphoto-181843094-640_adpp_is (1) (1).mp4
   istockphoto-181843093-640_adpp_is (1).mp4  (411.6 KB)
   istockphoto-181843094-640_adpp_is (1) (1).mp4  (471.3 KB)

Video 1: istockphoto-181843093-640_adpp_is (1).mp4
Video 2: istockphoto-181843094-640_adpp_is (1) (1).mp4


## Step 3 — Combine the Two Videos

In [ ]:
COMBINED_VIDEO = 'combined_input.mp4'
def normalize_video(src, dst, width=768, height=432, fps=25):
    cmd = [
        'ffmpeg', '-y', '-i', src,
        '-vf', f'scale={width}:{height},fps={fps}',
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
        '-an',
        dst
    ]
    subprocess.run(cmd, check=True, capture_output=True)

normalize_video(VIDEO_1, 'norm1.mp4')
normalize_video(VIDEO_2, 'norm2.mp4')
print('   Both videos normalised to 768×432 @ 25 fps')

with open('concat_list.txt', 'w') as f:
    f.write("file 'norm1.mp4'\nfile 'norm2.mp4'\n")

subprocess.run(
    ['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
     '-i', 'concat_list.txt', '-c', 'copy', COMBINED_VIDEO],
    check=True, capture_output=True
)

cap = cv2.VideoCapture(COMBINED_VIDEO)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_combined  = cap.get(cv2.CAP_PROP_FPS)
duration_s    = total_frames / fps_combined
cap.release()

print(f'  Combined video ready: {total_frames} frames  |  {duration_s:.1f}s  |  {fps_combined:.0f} fps')

   Both videos normalised to 768×432 @ 25 fps
  Combined video ready: 734 frames  |  29.4s  |  25 fps


## Step 4 — Load YOLOv8 Model

In [ ]:
model = YOLO('yolov8n.pt')
print(' YOLOv8n loaded')
print('Classes available:', list(model.names.values())[:20], '...')

 YOLOv8n loaded
Classes available: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow'] ...


In [ ]:
POLE_CLASSES        = {'traffic light', 'stop sign', 'parking meter', 'fire hydrant'}
POLE_ASPECT_THRESH  = 1.6
FALL_ASPECT_THRESH  = 0.9
DROP_THRESH_PX      = 40
MIN_BOX_AREA        = 300
CONF_THRESHOLD      = 0.30
COOLDOWN_FRAMES     = 30

@dataclass
class TrackState:
    last_ratio:  float = 0.0
    last_cy:     float = 0.0
    last_trigger: int  = -9999

@dataclass
class FallEvent:
    frame_idx: int
    timestamp: float
    track_id:  int
    bbox:      Tuple[int,int,int,int]
    reason:    str

def fmt_time(seconds: float) -> str:
    m = int(seconds) // 60
    s = seconds - m * 60
    return f'{m:02d}:{s:05.2f}'

def detect_falls(
    combined_path: str,
    output_path:   str,
    model,
    fps: float,
) -> List[FallEvent]:

    cap    = cv2.VideoCapture(combined_path)
    W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_in = cap.get(cv2.CAP_PROP_FPS) or fps

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(output_path, fourcc, fps_in, (W, H))

    track_states: dict[int, TrackState] = defaultdict(TrackState)
    fall_events:  List[FallEvent]       = []
    frame_idx = 0

    log_lines: List[str] = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        ts = frame_idx / fps_in

        results = model.track(
            frame,
            persist=True,
            conf=CONF_THRESHOLD,
            verbose=False,
            tracker='bytetrack.yaml'
        )[0]

        annotated = frame.copy()

        if results.boxes is not None and len(results.boxes):
            boxes   = results.boxes.xyxy.cpu().numpy()
            confs   = results.boxes.conf.cpu().numpy()
            cls_ids = results.boxes.cls.cpu().numpy().astype(int)
            ids_raw = results.boxes.id
            track_ids = ids_raw.cpu().numpy().astype(int) if ids_raw is not None \
                        else np.arange(len(boxes))

            for box, conf, cls_id, tid in zip(boxes, confs, cls_ids, track_ids):
                x1, y1, x2, y2 = map(int, box)
                bw = max(x2 - x1, 1)
                bh = max(y2 - y1, 1)
                area  = bw * bh
                ratio = bh / bw
                cy    = (y1 + y2) / 2
                label = model.names[cls_id]

                is_pole_class = label in POLE_CLASSES
                is_pole_shape = (ratio > POLE_ASPECT_THRESH) and (area > MIN_BOX_AREA)

                if not (is_pole_class or is_pole_shape):
                    continue

                st = track_states[tid]
                fallen = False
                reason = ''

                if st.last_ratio > POLE_ASPECT_THRESH and ratio < FALL_ASPECT_THRESH:
                    fallen = True
                    reason = f'ratio flip {st.last_ratio:.1f}→{ratio:.1f}'

                elif st.last_cy > 0 and (cy - st.last_cy) > DROP_THRESH_PX:
                    fallen = True
                    reason = f'drop {cy - st.last_cy:.0f}px'

                elif ratio < FALL_ASPECT_THRESH and is_pole_class:
                    fallen = True
                    reason = f'horizontal pole (ratio={ratio:.1f})'

                if fallen and (frame_idx - st.last_trigger) > COOLDOWN_FRAMES:
                    event = FallEvent(
                        frame_idx=frame_idx,
                        timestamp=ts,
                        track_id=tid,
                        bbox=(x1, y1, x2, y2),
                        reason=reason
                    )
                    fall_events.append(event)
                    st.last_trigger = frame_idx
                    log_lines.append(f'⚠ {fmt_time(ts)} | Track {tid} | {reason}')
                    print(f'   Fall detected at {fmt_time(ts)}  track={tid}  reason={reason}')

                st.last_ratio = ratio
                st.last_cy    = cy

                color = (0, 0, 255) if fallen else (0, 200, 0)
                thick = 3 if fallen else 2
                cv2.rectangle(annotated, (x1, y1), (x2, y2), color, thick)
                det_label = f'{label} {conf:.2f}'
                cv2.rectangle(annotated, (x1, y1 - 22), (x1 + len(det_label)*10, y1), color, -1)
                cv2.putText(annotated, det_label, (x1 + 2, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

        cv2.putText(annotated, fmt_time(ts), (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2)

        y_log = 60
        for line in log_lines[-6:]:
            cv2.putText(annotated, line, (10, y_log),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 100, 255), 1)
            y_log += 22

        out.write(annotated)
        frame_idx += 1

    cap.release()
    out.release()
    return fall_events

print(' Detector functions defined')

 Detector functions defined


## Step 6 — Run Detection on the Combined Video

In [ ]:
OUTPUT_RAW = 'output_raw.mp4'
OUTPUT_FINAL = 'fallen_pole_output.mp4'

print(' Running YOLOv8 tracking on combined video …')
events = detect_falls(
    combined_path = COMBINED_VIDEO,
    output_path   = OUTPUT_RAW,
    model         = model,
    fps           = fps_combined,
)

subprocess.run(
    ['ffmpeg', '-y', '-i', OUTPUT_RAW,
     '-vcodec', 'libx264', '-pix_fmt', 'yuv420p',
     OUTPUT_FINAL],
    check=True, capture_output=True
)

print(f'\n Detection complete — {len(events)} fall event(s) found')
print(f'   Output saved to: {OUTPUT_FINAL}')

 Running YOLOv8 tracking on combined video …
   Fall detected at 00:04.32  track=7  reason=drop 62px
   Fall detected at 00:04.60  track=1  reason=drop 72px
   Fall detected at 00:05.24  track=8  reason=drop 49px
   Fall detected at 00:05.84  track=28  reason=drop 70px
   Fall detected at 00:18.04  track=2  reason=drop 54px
   Fall detected at 00:18.04  track=3  reason=drop 46px
   Fall detected at 00:18.04  track=6  reason=drop 60px

 Detection complete — 7 fall event(s) found
   Output saved to: fallen_pole_output.mp4


## Step 7 — Summary Report

In [ ]:
print('='*55)
print('         FALLEN POLE DETECTION REPORT')
print('='*55)
if not events:
    print('    No fallen poles detected in this video.')
    print('   Tips to improve detection:')
    print('     • Lower CONF_THRESHOLD (currently 0.30)')
    print('     • Lower POLE_ASPECT_THRESH (currently 1.6)')
    print('     • Add more class names to POLE_CLASSES')
else:
    for i, ev in enumerate(events, 1):
        print(f'  [{i}]  Time : {fmt_time(ev.timestamp)}  (frame {ev.frame_idx})')
        print(f'        Track ID : {ev.track_id}')
        print(f'        Reason   : {ev.reason}')
        print(f'        BBox     : {ev.bbox}')
        print()
print('='*55)

         FALLEN POLE DETECTION REPORT
  [1]  Time : 00:04.32  (frame 108)
        Track ID : 7
        Reason   : drop 62px
        BBox     : (297, 35, 419, 371)

  [2]  Time : 00:04.60  (frame 115)
        Track ID : 1
        Reason   : drop 72px
        BBox     : (147, 170, 254, 425)

  [3]  Time : 00:05.24  (frame 131)
        Track ID : 8
        Reason   : drop 49px
        BBox     : (182, 152, 251, 372)

  [4]  Time : 00:05.84  (frame 146)
        Track ID : 28
        Reason   : drop 70px
        BBox     : (83, 60, 266, 353)

  [5]  Time : 00:18.04  (frame 451)
        Track ID : 2
        Reason   : drop 54px
        BBox     : (197, 216, 241, 343)

  [6]  Time : 00:18.04  (frame 451)
        Track ID : 3
        Reason   : drop 46px
        BBox     : (245, 222, 295, 356)

  [7]  Time : 00:18.04  (frame 451)
        Track ID : 6
        Reason   : drop 60px
        BBox     : (300, 229, 354, 380)



## Step 8 —  Play the Annotated Output Video in Colab

In [ ]:
def play_video_inline(path: str, width: int = 720):
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    html = f"""
    <video width="{width}" controls autoplay loop>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
      Your browser does not support HTML5 video.
    </video>
    """
    display(HTML(html))

print('  Rendering output video …')
play_video_inline(OUTPUT_FINAL)

  Rendering output video …


## Step 9 — Download the Output Video

In [ ]:
files.download(OUTPUT_FINAL)
print('⬇  Download started!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇  Download started!
